# Validation: 1D vs 2D HSQC Quantitative Comparison

This notebook validates the AI-enhanced 1D NMR method against conventional 2D ¹H–¹³C HSQC.

**Data sources:**
- `data/validation_hepg2/` — HepG2, 4 biological replicates (15 paired time points each)
- `data/validation_mcf7/` — MCF-7, 4 biological replicates
- `data/pa_experiment/` — HepG2 PA vs BSA, 8 samples, 450 time points (2-min resolution)

Each CSV contains 10 metabolite resonance signals (columns):
`glucose_108, scale_9, scale_41, scale_63, scale_18, scale_34, scale_81, scale_92, scale_96, scale_77`


In [ ]:
import os, glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, linregress

NB_DIR = Path.cwd()
DATA_DIR = NB_DIR / "data"
print(f"Data dir: {DATA_DIR.name}/")


## 1. Load Validation Data (HepG2 & MCF-7)

Each sample has two CSV files:
- `*_scale.csv` — 1D NOESY-derived intensities (after RH-Unet + BVLS)
- `*_HSQC.csv` — 2D HSQC-derived intensities (reference)


In [ ]:
def load_validation(folder):
    """Load paired 1D and 2D validation data from a folder."""
    scale_files = sorted(glob.glob(str(folder / "*_scale.csv")))
    hsqc_files = sorted(glob.glob(str(folder / "*_HSQC.csv")))

    scale_data, hsqc_data = [], []
    for sf, hf in zip(scale_files, hsqc_files):
        scale_data.append(pd.read_csv(sf).values)
        hsqc_data.append(pd.read_csv(hf).values)

    labels = pd.read_csv(scale_files[0]).columns.tolist()
    return np.array(scale_data), np.array(hsqc_data), labels

# Load HepG2
hepg_scale, hepg_hsqc, labels = load_validation(DATA_DIR / "validation_hepg2")
# Load MCF-7
mcf_scale, mcf_hsqc, _ = load_validation(DATA_DIR / "validation_mcf7")

print(f"HepG2: scale {hepg_scale.shape}, hsqc {hepg_hsqc.shape}")
print(f"MCF-7: scale {mcf_scale.shape}, hsqc {mcf_hsqc.shape}")
print(f"Labels: {labels}")


## 2. Per-Metabolite Agreement Statistics

For each metabolite, compute Pearson r, Bland-Altman bias, and 95% limits of agreement.


In [ ]:
def compute_stats(scale, hsqc, labels):
    """Compute Pearson r, bias, LoA for each metabolite."""
    results = []
    for i, label in enumerate(labels):
        s = scale[:, :, i].flatten()
        h = hsqc[:, :, i].flatten()
        r, p = pearsonr(s, h)
        diff = s - h
        bias = np.mean(diff)
        loa = 1.96 * np.std(diff)
        results.append({
            'metabolite': label,
            'r': r,
            'p': p,
            'bias_pct': bias * 100,
            'loa_pct': loa * 100,
        })
    return pd.DataFrame(results)

hepg_stats = compute_stats(hepg_scale, hepg_hsqc, labels)
mcf_stats = compute_stats(mcf_scale, mcf_hsqc, labels)

print("=== HepG2 ===")
print(hepg_stats.round(3).to_string(index=False))
print()
print("=== MCF-7 ===")
print(mcf_stats.round(3).to_string(index=False))


## 3. Temporal Profiles: 1D vs 2D (Figure 4)

Plot paired time courses for each metabolite signal.


In [ ]:
def plot_validation(scale, hsqc, labels, title_prefix=""):
    n = len(labels)
    fig, axes = plt.subplots((n+3)//4, 4, figsize=(16, 3*((n+3)//4)))
    for i, label in enumerate(labels):
        ax = axes.flat[i]
        for s in scale:
            ax.plot(s[:, i], color='#1f77b4', alpha=0.4, lw=0.8)
        for h in hsqc:
            ax.plot(h[:, i], color='#d62728', alpha=0.4, lw=0.8)
        ax.set_title(label, fontsize=9)
        ax.set_xlabel('Time point')
        ax.set_ylabel('Normalized intensity')
    plt.suptitle(f"{title_prefix}1D (blue) vs 2D HSQC (red)", y=1.02, fontsize=12)
    plt.tight_layout()
    plt.show()

plot_validation(hepg_scale, hepg_hsqc, labels, "HepG2: ")


## 4. PA vs BSA Experiment (Figure 5)

Time-course of 8 metabolite signals under palmitic acid (PA) treatment vs BSA control.


In [ ]:
# Load PA experiment data
pa_dir = DATA_DIR / "pa_experiment"
bsa_files = sorted(glob.glob(str(pa_dir / "*_bsa_*.csv")))
pa_files = sorted(glob.glob(str(pa_dir / "*_pa_*.csv")))

bsa_data = np.array([pd.read_csv(f).values for f in bsa_files])
pa_data = np.array([pd.read_csv(f).values for f in pa_files])

print(f"BSA: {bsa_data.shape}, PA: {pa_data.shape}")
print(f"Time points: {bsa_data.shape[1]} (2-min intervals)")

# All 10 metabolite column names (in CSV order)
all_names = [
    'glucose@5.23', 'glucose@3.23', 'choline@3.21', 'betaine@3.24',
    'lactate@4.10', 'unknown@4.16', 'lipid@1.28', 'lipid@1.29',
    'lactate@1.32', 'lipid@1.26'
]

# Pretty names matching reference figure style
pretty_names = [
    'glucose@5.23 ppm', 'glucose@3.23 ppm', 'choline@3.21 ppm', 'betaine@3.24 ppm',
    'lactate@4.10 ppm', 'unknown@4.16 ppm',
    '(-CH$_2$–)$_n$@1.28 ppm', '(-CH$_2$–)$_n$@1.29 ppm',
    'lactate@1.32 ppm', '(-CH$_2$–)$_n$@1.26 ppm'
]

# Exclude betaine (idx 3) and unknown@4.16 (idx 5)
INCLUDE_IDX = [0, 1, 2, 4, 6, 7, 8, 9]
INCLUDE_NAMES = [pretty_names[i] for i in INCLUDE_IDX]
PANEL_LABELS = ['(a)', '(b)', '(c)', '(d)', '(e)', '(f)', '(g)', '(h)']
print(f"Plotting {len(INCLUDE_NAMES)} signals")


In [ ]:
# Plot mean ± SD for each included metabolite
# Normalize each sample to its own t0 = 1.0
bsa_norm = bsa_data / bsa_data[:, 0:1, :]
pa_norm  = pa_data  / pa_data[:, 0:1, :]

t_min = np.arange(bsa_norm.shape[1]) * 2  # 2 min per point

# Publication-style figure matching reference
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.subplots_adjust(top=0.88, hspace=0.35, wspace=0.30)

for plot_i, data_i in enumerate(INCLUDE_IDX):
    ax = axes.flat[plot_i]
    # BSA
    b_mean = bsa_norm[:, :, data_i].mean(axis=0)
    b_std  = bsa_norm[:, :, data_i].std(axis=0)
    ax.plot(t_min, b_mean, color='#2166AC', lw=2.0, label='BSA control' if plot_i == 0 else '')
    ax.fill_between(t_min, b_mean-b_std, b_mean+b_std, alpha=0.25, color='#2166AC')
    # PA
    p_mean = pa_norm[:, :, data_i].mean(axis=0)
    p_std  = pa_norm[:, :, data_i].std(axis=0)
    ax.plot(t_min, p_mean, color='#B2182B', lw=2.0, label='PA treatment' if plot_i == 0 else '')
    ax.fill_between(t_min, p_mean-p_std, p_mean+p_std, alpha=0.25, color='#B2182B')
    # Panel label
    ax.text(-0.12, 1.05, PANEL_LABELS[plot_i], transform=ax.transAxes,
            fontsize=14, fontweight='bold', va='top')
    ax.set_title(INCLUDE_NAMES[plot_i], fontsize=11, pad=8)
    ax.set_xlabel('Time (min)', fontsize=11)
    ax.set_ylabel('Normalized Intensity', fontsize=11)
    ax.tick_params(labelsize=10)
    # Clean spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(1.2)
    ax.spines['bottom'].set_linewidth(1.2)
    ax.tick_params(direction='out', width=1.2)

# Global legend at top
handles = [plt.Line2D([], [], color='#2166AC', lw=2.5),
           plt.Line2D([], [], color='#B2182B', lw=2.5)]
fig.legend(handles, ['BSA control', 'PA treatment'],
           loc='upper center', ncol=2, frameon=False,
           fontsize=14, bbox_to_anchor=(0.5, 0.99))

plt.show()


## 5. Statistical Comparison: Slope Analysis

Compare PA vs BSA using linear regression slope for each metabolite.


In [ ]:
from scipy.stats import ttest_ind

results = []
x = np.arange(bsa_data.shape[1])
for plot_i, data_i in enumerate(INCLUDE_IDX):
    name = INCLUDE_NAMES[plot_i]
    bsa_slopes = [linregress(x, s[:, data_i]).slope for s in bsa_data]
    pa_slopes = [linregress(x, s[:, data_i]).slope for s in pa_data]
    t, p = ttest_ind(bsa_slopes, pa_slopes, equal_var=False)
    results.append({
        'metabolite': name,
        'BSA_slope': np.mean(bsa_slopes),
        'PA_slope': np.mean(pa_slopes),
        'ratio': np.mean(pa_slopes) / np.mean(bsa_slopes) if np.mean(bsa_slopes) != 0 else np.nan,
        't': t,
        'p': p
    })

slope_df = pd.DataFrame(results)
print(slope_df.round(4).to_string(index=False))
